# 🎬 Video TTS Synchronizer v2.0 - Google Colab Edition
## Sistema Inteligente de Subtítulos + TTS Multiidioma

### 🆕 Novedades v2.0:
- ✅ **Sistema inteligente de obtención de subtítulos**:
  1. Intenta subtítulos manuales de YouTube en idioma objetivo
  2. Intenta transcripción automática de YouTube en idioma objetivo
  3. Descarga subtítulos en idioma original y traduce
  4. Usa Whisper como último recurso
- ✅ **Traducción automática** con Google Translate
- ✅ **Múltiples motores TTS** gratuitos
- ✅ **Descarga de YouTube** optimizada
- ✅ **Sincronización perfecta** con extensión de frames

---

## 📦 Paso 1: Instalación de Dependencias

In [ ]:
%%capture
# Instalar todas las dependencias necesarias
!apt-get update
!apt-get install -y ffmpeg

# Librerías de Python
!pip install -q yt-dlp
!pip install -q openai-whisper
!pip install -q pysrt
!pip install -q pydub
!pip install -q gTTS
!pip install -q edge-tts
!pip install -q deep-translator  # Para traducción automática
!pip install -q langdetect  # Para detectar idioma

print("✅ Todas las dependencias instaladas")

## 🛠️ Paso 2: Importar Librerías

In [ ]:
import os
import sys
import subprocess
import asyncio
import re
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional
import json

import pysrt
import whisper
from pydub import AudioSegment
from gtts import gTTS
import edge_tts
from deep_translator import GoogleTranslator
from langdetect import detect
from IPython.display import Video, Audio, display, HTML, Markdown

print("✅ Librerías importadas correctamente")

## 📥 Paso 3: Sistema Inteligente de Subtítulos

In [ ]:
class IntelligentSubtitleManager:
    """
    Gestor inteligente de subtítulos con múltiples estrategias de obtención
    """
    
    def __init__(self, youtube_url: str, target_lang: str = "es"):
        self.youtube_url = youtube_url
        self.target_lang = target_lang
        self.video_id = self._extract_video_id(youtube_url)
        
    def _extract_video_id(self, url: str) -> str:
        """Extrae el ID del video de YouTube"""
        patterns = [
            r'(?:v=|\/)([0-9A-Za-z_-]{11}).*',
            r'(?:embed\/|v\/|youtu.be\/)([0-9A-Za-z_-]{11})'
        ]
        for pattern in patterns:
            match = re.search(pattern, url)
            if match:
                return match.group(1)
        return ""
    
    def get_available_subtitles(self) -> dict:
        """
        Obtiene lista de subtítulos disponibles del video
        """
        print("🔍 Buscando subtítulos disponibles en YouTube...")
        
        try:
            result = subprocess.run([
                'yt-dlp',
                '--list-subs',
                '--skip-download',
                self.youtube_url
            ], capture_output=True, text=True)
            
            output = result.stdout
            
            # Parsear subtítulos disponibles
            available_subs = {
                'manual': [],
                'automatic': []
            }
            
            # Buscar secciones de subtítulos
            if 'Available subtitles' in output:
                lines = output.split('\n')
                for i, line in enumerate(lines):
                    if 'Language' in line and 'formats' in line.lower():
                        # Siguiente líneas contienen idiomas
                        for j in range(i+1, len(lines)):
                            if lines[j].strip() and not lines[j].startswith('Available'):
                                parts = lines[j].split()
                                if parts:
                                    lang_code = parts[0]
                                    available_subs['manual'].append(lang_code)
                            else:
                                break
            
            if 'Available automatic captions' in output:
                lines = output.split('\n')
                for i, line in enumerate(lines):
                    if 'Available automatic captions' in line:
                        for j in range(i+1, len(lines)):
                            if lines[j].strip() and not lines[j].startswith('['):
                                parts = lines[j].split()
                                if parts:
                                    lang_code = parts[0]
                                    available_subs['automatic'].append(lang_code)
                            else:
                                break
            
            return available_subs
            
        except Exception as e:
            print(f"⚠️  Error al listar subtítulos: {e}")
            return {'manual': [], 'automatic': []}
    
    def download_youtube_subtitle(self, lang_code: str, subtitle_type: str = "manual") -> Optional[str]:
        """
        Descarga subtítulos de YouTube
        
        Args:
            lang_code: Código de idioma (es, en, de, etc.)
            subtitle_type: 'manual' o 'automatic'
        """
        try:
            output_template = f"youtube_sub_{lang_code}"
            
            cmd = [
                'yt-dlp',
                '--write-sub' if subtitle_type == 'manual' else '--write-auto-sub',
                '--sub-lang', lang_code,
                '--skip-download',
                '--sub-format', 'srt',
                '-o', output_template,
                self.youtube_url
            ]
            
            result = subprocess.run(cmd, capture_output=True, text=True)
            
            # Buscar archivo descargado
            srt_file = f"{output_template}.{lang_code}.srt"
            if os.path.exists(srt_file):
                print(f"✅ Subtítulos descargados: {srt_file}")
                return srt_file
            
            return None
            
        except Exception as e:
            print(f"⚠️  Error descargando subtítulos: {e}")
            return None
    
    def translate_subtitle_file(self, srt_path: str, source_lang: str, target_lang: str) -> str:
        """
        Traduce un archivo SRT completo
        """
        print(f"🌐 Traduciendo subtítulos de {source_lang} a {target_lang}...")
        
        try:
            # Leer subtítulos originales
            subs = pysrt.open(srt_path, encoding='utf-8')
            
            # Inicializar traductor
            translator = GoogleTranslator(source=source_lang, target=target_lang)
            
            # Traducir cada subtítulo
            total = len(subs)
            for i, sub in enumerate(subs):
                if sub.text.strip():
                    try:
                        translated = translator.translate(sub.text)
                        sub.text = translated
                        
                        # Mostrar progreso
                        if (i + 1) % 10 == 0:
                            print(f"   Traduciendo: {i+1}/{total}")
                    except Exception as e:
                        print(f"   ⚠️  Error traduciendo línea {i+1}: {e}")
                        # Mantener texto original si falla
            
            # Guardar subtítulos traducidos
            translated_path = srt_path.replace('.srt', f'_translated_{target_lang}.srt')
            subs.save(translated_path, encoding='utf-8')
            
            print(f"✅ Traducción completada: {translated_path}")
            return translated_path
            
        except Exception as e:
            print(f"❌ Error en traducción: {e}")
            return srt_path  # Devolver original si falla
    
    def detect_video_language(self) -> str:
        """
        Intenta detectar el idioma del video descargando un fragmento
        """
        print("🔍 Detectando idioma del video...")
        
        try:
            # Obtener info del video
            result = subprocess.run([
                'yt-dlp',
                '--dump-json',
                '--skip-download',
                self.youtube_url
            ], capture_output=True, text=True)
            
            info = json.loads(result.stdout)
            
            # Intentar obtener idioma del video
            if 'language' in info:
                lang = info['language']
                print(f"   Idioma detectado: {lang}")
                return lang
            
            # Si no hay info, intentar con título/descripción
            if 'title' in info:
                try:
                    detected = detect(info['title'])
                    print(f"   Idioma detectado (por título): {detected}")
                    return detected
                except:
                    pass
            
            print("   ⚠️  No se pudo detectar idioma, asumiendo inglés")
            return 'en'
            
        except Exception as e:
            print(f"   ⚠️  Error detectando idioma: {e}")
            return 'en'
    
    def get_subtitle_intelligent(self, use_whisper_translation: bool = False) -> str:
        """
        Estrategia inteligente para obtener subtítulos:
        
        1. Intenta subtítulos manuales en idioma objetivo
        2. Intenta subtítulos automáticos en idioma objetivo
        3. Descarga en idioma original y traduce
        4. Usa Whisper (con o sin traducción)
        """
        print("\n" + "="*80)
        print("🎯 SISTEMA INTELIGENTE DE OBTENCIÓN DE SUBTÍTULOS")
        print("="*80)
        print(f"Idioma objetivo: {self.target_lang}")
        print()
        
        # Obtener subtítulos disponibles
        available = self.get_available_subtitles()
        
        print(f"\n📋 Subtítulos disponibles:")
        print(f"   Manuales: {', '.join(available['manual']) if available['manual'] else 'Ninguno'}")
        print(f"   Automáticos: {', '.join(available['automatic']) if available['automatic'] else 'Ninguno'}")
        
        # ESTRATEGIA 1: Subtítulos manuales en idioma objetivo
        print(f"\n🎯 Estrategia 1: Buscando subtítulos manuales en '{self.target_lang}'...")
        if self.target_lang in available['manual']:
            subtitle_file = self.download_youtube_subtitle(self.target_lang, 'manual')
            if subtitle_file:
                print("✅ ¡Éxito! Usando subtítulos manuales.")
                return subtitle_file
        else:
            print(f"   ⚠️  No hay subtítulos manuales en '{self.target_lang}'")
        
        # ESTRATEGIA 2: Subtítulos automáticos en idioma objetivo
        print(f"\n🎯 Estrategia 2: Buscando transcripción automática en '{self.target_lang}'...")
        if self.target_lang in available['automatic']:
            subtitle_file = self.download_youtube_subtitle(self.target_lang, 'automatic')
            if subtitle_file:
                print("✅ ¡Éxito! Usando transcripción automática.")
                return subtitle_file
        else:
            print(f"   ⚠️  No hay transcripción automática en '{self.target_lang}'")
        
        # ESTRATEGIA 3: Descargar en idioma original y traducir
        print(f"\n🎯 Estrategia 3: Descargando en idioma original y traduciendo...")
        
        # Detectar idioma del video
        original_lang = self.detect_video_language()
        
        # Intentar descargar en idioma original
        original_subtitle = None
        
        # Primero manuales
        if original_lang in available['manual']:
            original_subtitle = self.download_youtube_subtitle(original_lang, 'manual')
        
        # Si no, automáticos
        if not original_subtitle and original_lang in available['automatic']:
            original_subtitle = self.download_youtube_subtitle(original_lang, 'automatic')
        
        # Intentar con idioma más común si original no funciona
        if not original_subtitle:
            for lang in ['en', 'es', 'de', 'fr', 'it', 'pt']:
                if lang in available['manual']:
                    original_subtitle = self.download_youtube_subtitle(lang, 'manual')
                    original_lang = lang
                    break
                elif lang in available['automatic']:
                    original_subtitle = self.download_youtube_subtitle(lang, 'automatic')
                    original_lang = lang
                    break
        
        if original_subtitle:
            print(f"   ✅ Subtítulos descargados en '{original_lang}'")
            print(f"   🌐 Traduciendo a '{self.target_lang}'...")
            translated_file = self.translate_subtitle_file(
                original_subtitle, 
                original_lang, 
                self.target_lang
            )
            print("✅ ¡Éxito! Usando subtítulos traducidos.")
            return translated_file
        else:
            print("   ⚠️  No se pudieron descargar subtítulos de YouTube")
        
        # ESTRATEGIA 4: Whisper
        print(f"\n🎯 Estrategia 4: Usando Whisper para transcribir...")
        print("   ⚠️  Esto puede tomar varios minutos...")
        
        # Descargar video
        video_path = self.download_video()
        
        if use_whisper_translation:
            print(f"   🌐 Whisper transcribirá y traducirá a '{self.target_lang}'")
            subtitle_file = self.transcribe_with_whisper_translation(
                video_path, 
                self.target_lang
            )
        else:
            print(f"   📝 Whisper transcribirá en idioma original")
            subtitle_file = self.transcribe_with_whisper(
                video_path, 
                original_lang
            )
            
            # Traducir si es necesario
            if original_lang != self.target_lang:
                print(f"   🌐 Traduciendo a '{self.target_lang}'...")
                subtitle_file = self.translate_subtitle_file(
                    subtitle_file,
                    original_lang,
                    self.target_lang
                )
        
        print("✅ ¡Éxito! Usando transcripción de Whisper.")
        return subtitle_file
    
    def download_video(self) -> str:
        """Descarga el video de YouTube"""
        print("📥 Descargando video...")
        
        output_path = "youtube_video.mp4"
        
        subprocess.run([
            'yt-dlp',
            '-f', 'best[ext=mp4]',
            '-o', output_path,
            self.youtube_url
        ], check=True)
        
        print(f"✅ Video descargado: {output_path}")
        return output_path
    
    def transcribe_with_whisper(self, video_path: str, language: str, 
                               model_size: str = "base") -> str:
        """Transcribe con Whisper en idioma original"""
        print(f"🎤 Transcribiendo con Whisper ({model_size})...")
        
        # Extraer audio
        audio_path = "temp_audio.wav"
        subprocess.run([
            'ffmpeg', '-y', '-i', video_path,
            '-vn', '-acodec', 'pcm_s16le',
            '-ar', '16000', '-ac', '1',
            audio_path
        ], capture_output=True, check=True)
        
        # Cargar modelo
        model = whisper.load_model(model_size)
        
        # Transcribir
        result = model.transcribe(
            audio_path,
            language=language,
            task="transcribe"
        )
        
        # Generar SRT
        srt_path = "whisper_transcription.srt"
        self._save_whisper_result_to_srt(result, srt_path)
        
        os.remove(audio_path)
        
        return srt_path
    
    def transcribe_with_whisper_translation(self, video_path: str, target_lang: str,
                                           model_size: str = "base") -> str:
        """Transcribe y traduce con Whisper"""
        print(f"🎤 Transcribiendo y traduciendo con Whisper ({model_size})...")
        
        # Extraer audio
        audio_path = "temp_audio.wav"
        subprocess.run([
            'ffmpeg', '-y', '-i', video_path,
            '-vn', '-acodec', 'pcm_s16le',
            '-ar', '16000', '-ac', '1',
            audio_path
        ], capture_output=True, check=True)
        
        # Cargar modelo
        model = whisper.load_model(model_size)
        
        # Transcribir y traducir a inglés (Whisper solo traduce a inglés)
        result = model.transcribe(
            audio_path,
            task="translate"  # Traduce a inglés
        )
        
        # Generar SRT temporal
        temp_srt = "whisper_english.srt"
        self._save_whisper_result_to_srt(result, temp_srt)
        
        os.remove(audio_path)
        
        # Si el objetivo no es inglés, traducir
        if target_lang != 'en':
            print(f"   🌐 Traduciendo de inglés a {target_lang}...")
            final_srt = self.translate_subtitle_file(temp_srt, 'en', target_lang)
            return final_srt
        
        return temp_srt
    
    def _save_whisper_result_to_srt(self, result: dict, output_path: str):
        """Guarda resultado de Whisper en formato SRT"""
        with open(output_path, 'w', encoding='utf-8') as f:
            for i, segment in enumerate(result['segments'], 1):
                start = self._format_timestamp(segment['start'])
                end = self._format_timestamp(segment['end'])
                text = segment['text'].strip()
                
                f.write(f"{i}\n")
                f.write(f"{start} --> {end}\n")
                f.write(f"{text}\n\n")
    
    def _format_timestamp(self, seconds: float) -> str:
        """Formatea timestamp para SRT"""
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        millis = int((seconds % 1) * 1000)
        return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"


print("✅ Sistema inteligente de subtítulos definido")

## 🎤 Paso 4: Motores TTS

In [ ]:
class TTSEngine:
    """Clase base para motores TTS"""
    
    @staticmethod
    def list_available_engines():
        print("\n" + "="*80)
        print("🎤 MOTORES TTS GRATUITOS DISPONIBLES")
        print("="*80)
        
        print("\n1️⃣  edge-tts (Microsoft Edge TTS) ⭐ RECOMENDADO")
        print("   - Gratuito, alta calidad")
        print("   - Voces neuronales")
        print("   - 100+ voces en 50+ idiomas")
        
        print("\n2️⃣  gTTS (Google Text-to-Speech)")
        print("   - Gratuito, simple")
        print("   - Calidad media")
        print("   - Muchos idiomas")
        
        print("\n" + "="*80)


class GTTSEngine:
    @staticmethod
    def generate(text: str, output_file: str, lang: str = "es") -> int:
        tts = gTTS(text=text, lang=lang, slow=False)
        tts.save(output_file)
        audio = AudioSegment.from_file(output_file)
        return len(audio)


class EdgeTTSEngine:
    @staticmethod
    async def generate_async(text: str, output_file: str, voice: str) -> int:
        communicate = edge_tts.Communicate(text, voice)
        await communicate.save(output_file)
        audio = AudioSegment.from_file(output_file)
        return len(audio)
    
    @staticmethod
    def generate(text: str, output_file: str, voice: str) -> int:
        return asyncio.run(EdgeTTSEngine.generate_async(text, output_file, voice))
    
    @staticmethod
    def get_recommended_voices():
        return {
            'es': [
                ('es-ES-AlvaroNeural', 'España - Hombre ⭐'),
                ('es-MX-DaliaNeural', 'México - Mujer ⭐'),
                ('es-ES-ElviraNeural', 'España - Mujer'),
                ('es-AR-ElenaNeural', 'Argentina - Mujer'),
            ],
            'en': [
                ('en-US-AriaNeural', 'USA - Mujer ⭐'),
                ('en-GB-SoniaNeural', 'UK - Mujer ⭐'),
                ('en-US-GuyNeural', 'USA - Hombre'),
                ('en-GB-RyanNeural', 'UK - Hombre'),
            ],
            'de': [
                ('de-DE-KatjaNeural', 'Alemania - Mujer ⭐'),
                ('de-DE-ConradNeural', 'Alemania - Hombre ⭐'),
                ('de-AT-IngridNeural', 'Austria - Mujer'),
            ]
        }


print("✅ Motores TTS definidos")

## 🎬 Paso 5: Procesador Principal

In [ ]:
@dataclass
class SubtitleSegment:
    index: int
    start_ms: int
    end_ms: int
    duration_ms: int
    text: str


class VideoTTSProcessor:
    def __init__(self, video_path: str, subtitle_path: str, output_path: str,
                 tts_engine: str = "edge", tts_voice: str = "es-ES-AlvaroNeural",
                 tts_lang: str = "es"):
        self.video_path = Path(video_path)
        self.subtitle_path = Path(subtitle_path)
        self.output_path = Path(output_path)
        self.tts_engine = tts_engine
        self.tts_voice = tts_voice
        self.tts_lang = tts_lang
        self.temp_dir = Path("temp_processing")
        self.temp_dir.mkdir(exist_ok=True)
    
    def parse_subtitles(self) -> List[SubtitleSegment]:
        print(f"📖 Leyendo: {self.subtitle_path}")
        subs = pysrt.open(str(self.subtitle_path), encoding='utf-8')
        
        segments = []
        for i, sub in enumerate(subs):
            start_ms = (sub.start.hours * 3600000 + sub.start.minutes * 60000 +
                       sub.start.seconds * 1000 + sub.start.milliseconds)
            end_ms = (sub.end.hours * 3600000 + sub.end.minutes * 60000 +
                     sub.end.seconds * 1000 + sub.end.milliseconds)
            
            segments.append(SubtitleSegment(
                index=i,
                start_ms=start_ms,
                end_ms=end_ms,
                duration_ms=end_ms - start_ms,
                text=sub.text.replace('\n', ' ').strip()
            ))
        
        print(f"✅ {len(segments)} segmentos")
        return segments
    
    def generate_tts_audio(self, text: str, output_file: Path) -> int:
        print(f"🎤 TTS: '{text[:50]}...'")
        
        if self.tts_engine == "gtts":
            duration = GTTSEngine.generate(text, str(output_file), self.tts_lang)
        elif self.tts_engine == "edge":
            duration = EdgeTTSEngine.generate(text, str(output_file), self.tts_voice)
        else:
            raise ValueError(f"Motor TTS desconocido: {self.tts_engine}")
        
        print(f"   ⏱️  {duration/1000:.2f}s")
        return duration
    
    def generate_all_tts(self, segments: List[SubtitleSegment]):
        print("\n🎵 Generando TTS...")
        audio_files = []
        
        for segment in segments:
            if not segment.text.strip():
                continue
            
            audio_file = self.temp_dir / f"tts_{segment.index:04d}.mp3"
            duration = self.generate_tts_audio(segment.text, audio_file)
            audio_files.append((audio_file, duration, segment))
        
        return audio_files
    
    def create_adjusted_timeline(self, segments, audio_files):
        print("\n⚙️  Timeline...")
        timeline = []
        current_time = 0
        
        for i, (audio_file, tts_duration_ms, segment) in enumerate(audio_files):
            padding = 200
            final_duration = max(tts_duration_ms, segment.duration_ms) + padding
            
            timeline.append({
                'index': i,
                'start_original': segment.start_ms,
                'end_original': segment.end_ms,
                'start_new': current_time,
                'end_new': current_time + final_duration,
                'duration_new': final_duration,
                'audio_file': audio_file,
                'text': segment.text,
                'extended': tts_duration_ms > segment.duration_ms,
                'extension_ms': final_duration - segment.duration_ms if tts_duration_ms > segment.duration_ms else 0
            })
            
            current_time += final_duration
        
        print(f"✅ Duración: {current_time/1000:.2f}s")
        return timeline
    
    def create_video_segments(self, timeline):
        print("\n🎬 Procesando video...")
        video_segments = []
        
        for entry in timeline:
            segment_file = self.temp_dir / f"video_{entry['index']:04d}.mp4"
            start_s = entry['start_original'] / 1000
            orig_dur_s = (entry['end_original'] - entry['start_original']) / 1000
            new_dur_s = entry['duration_new'] / 1000
            
            if entry['extended']:
                print(f"   📌 Seg {entry['index']}: Extendiendo...")
                
                temp_seg = self.temp_dir / f"temp_{entry['index']:04d}.mp4"
                subprocess.run([
                    'ffmpeg', '-y', '-i', str(self.video_path),
                    '-ss', str(start_s), '-t', str(orig_dur_s),
                    '-c', 'copy', str(temp_seg)
                ], capture_output=True, check=True)
                
                frame = self.temp_dir / f"frame_{entry['index']:04d}.png"
                subprocess.run([
                    'ffmpeg', '-y', '-sseof', '-0.1', '-i', str(temp_seg),
                    '-vframes', '1', str(frame)
                ], capture_output=True, check=True)
                
                ext_s = entry['extension_ms'] / 1000
                frozen = self.temp_dir / f"frozen_{entry['index']:04d}.mp4"
                subprocess.run([
                    'ffmpeg', '-y', '-loop', '1', '-i', str(frame),
                    '-t', str(ext_s), '-c:v', 'libx264',
                    '-pix_fmt', 'yuv420p', str(frozen)
                ], capture_output=True, check=True)
                
                concat_list = self.temp_dir / f"concat_{entry['index']:04d}.txt"
                with open(concat_list, 'w') as f:
                    f.write(f"file '{temp_seg.absolute()}'\n")
                    f.write(f"file '{frozen.absolute()}'\n")
                
                subprocess.run([
                    'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
                    '-i', str(concat_list), '-c', 'copy', str(segment_file)
                ], capture_output=True, check=True)
            else:
                subprocess.run([
                    'ffmpeg', '-y', '-i', str(self.video_path),
                    '-ss', str(start_s), '-t', str(new_dur_s),
                    '-c', 'copy', str(segment_file)
                ], capture_output=True, check=True)
            
            video_segments.append(segment_file)
        
        return video_segments
    
    def combine_audio(self, timeline):
        print("\n🔊 Combinando audio...")
        combined = AudioSegment.empty()
        
        for entry in timeline:
            audio = AudioSegment.from_file(str(entry['audio_file']))
            target = entry['duration_new']
            
            if len(audio) < target:
                audio = audio + AudioSegment.silent(duration=target - len(audio))
            
            combined += audio
        
        final_audio = self.temp_dir / "final_audio.mp3"
        combined.export(str(final_audio), format="mp3")
        return final_audio
    
    def combine_video(self, video_segments):
        print("\n🎬 Combinando video...")
        concat_list = self.temp_dir / "concat.txt"
        
        with open(concat_list, 'w') as f:
            for seg in video_segments:
                f.write(f"file '{seg.absolute()}'\n")
        
        combined = self.temp_dir / "combined_video.mp4"
        subprocess.run([
            'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
            '-i', str(concat_list), '-c', 'copy', str(combined)
        ], capture_output=True, check=True)
        
        return combined
    
    def merge_final(self, video_file, audio_file):
        print(f"\n🎞️  Generando: {self.output_path}")
        
        subprocess.run([
            'ffmpeg', '-y', '-i', str(video_file), '-i', str(audio_file),
            '-c:v', 'copy', '-c:a', 'aac',
            '-map', '0:v:0', '-map', '1:a:0',
            str(self.output_path)
        ], check=True)
        
        print(f"✅ ¡Completado!")
    
    def process(self):
        print("\n" + "="*80)
        print(f"🎬 PROCESANDO CON {self.tts_engine.upper()}")
        print("="*80)
        
        segments = self.parse_subtitles()
        audio_files = self.generate_all_tts(segments)
        timeline = self.create_adjusted_timeline(segments, audio_files)
        video_segments = self.create_video_segments(timeline)
        final_audio = self.combine_audio(timeline)
        final_video = self.combine_video(video_segments)
        self.merge_final(final_video, final_audio)
        
        print("\n✨ ¡Proceso completado!\n")


print("✅ Procesador definido")

## 🎯 Paso 6: CONFIGURACIÓN - Edita estos parámetros

In [ ]:
# ============================================================================
# CONFIGURA AQUÍ TUS PARÁMETROS
# ============================================================================

# 📹 VIDEO DE ENTRADA
USE_YOUTUBE = True  # True para YouTube, False para video local
YOUTUBE_URL = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"  # Cambia esto
LOCAL_VIDEO_PATH = "/content/mi_video.mp4"  # Si USE_YOUTUBE = False

# 📝 SUBTÍTULOS
# Opciones para obtener subtítulos:
# - "youtube_auto": Sistema inteligente (YouTube → Traducción → Whisper)
# - "youtube_manual": Solo subtítulos manuales de YouTube
# - "whisper": Solo Whisper (transcribe el video)
# - "local": Usar archivo SRT local

SUBTITLE_SOURCE = "youtube_auto"  # youtube_auto, youtube_manual, whisper, local
LOCAL_SUBTITLE_PATH = "/content/subtitulos.srt"  # Si SUBTITLE_SOURCE = "local"

# 🌐 IDIOMA
TARGET_LANGUAGE = "es"  # es, en, de, fr, it, pt, etc.

# 🎤 WHISPER (solo si se usa)
WHISPER_MODEL = "base"  # tiny, base, small, medium, large
USE_WHISPER_TRANSLATION = False  # True para que Whisper intente traducir

# 🔊 MOTOR TTS
TTS_ENGINE = "edge"  # "edge" o "gtts"

# Para gTTS
GTTS_LANG = "es"  # es, en, de

# Para edge-tts (RECOMENDADO)
# Voces recomendadas por idioma:
EDGE_VOICES = {
    'es': 'es-ES-AlvaroNeural',      # España - Hombre
    'en': 'en-US-AriaNeural',        # USA - Mujer
    'de': 'de-DE-KatjaNeural',       # Alemania - Mujer
}

EDGE_VOICE = EDGE_VOICES.get(TARGET_LANGUAGE, 'es-ES-AlvaroNeural')

# O especifica manualmente:
# EDGE_VOICE = "es-MX-DaliaNeural"  # México - Mujer
# EDGE_VOICE = "en-GB-SoniaNeural"  # UK - Mujer
# EDGE_VOICE = "de-DE-ConradNeural" # Alemania - Hombre

# 📤 SALIDA
OUTPUT_PATH = "/content/video_con_tts.mp4"

# ============================================================================

print("✅ Configuración lista")
print(f"\n📋 RESUMEN:")
print(f"   Fuente: {'YouTube' if USE_YOUTUBE else 'Local'}")
print(f"   Subtítulos: {SUBTITLE_SOURCE}")
print(f"   Idioma objetivo: {TARGET_LANGUAGE}")
print(f"   Motor TTS: {TTS_ENGINE}")
if TTS_ENGINE == "edge":
    print(f"   Voz: {EDGE_VOICE}")
print(f"   Salida: {OUTPUT_PATH}")

## ▶️ Paso 7: EJECUTAR - Procesar Video

In [ ]:
# Determinar video y subtítulos
video_path = None
subtitle_path = None

# PASO 1: Obtener video
if USE_YOUTUBE:
    print("📥 Descargando video de YouTube...")
    
    # Descargar video
    video_path = "youtube_video.mp4"
    subprocess.run([
        'yt-dlp',
        '-f', 'best[ext=mp4]',
        '-o', video_path,
        YOUTUBE_URL
    ], check=True)
    
    print(f"✅ Video descargado: {video_path}")
else:
    video_path = LOCAL_VIDEO_PATH
    print(f"📁 Usando video local: {video_path}")

# PASO 2: Obtener subtítulos
if SUBTITLE_SOURCE == "local":
    subtitle_path = LOCAL_SUBTITLE_PATH
    print(f"📄 Usando subtítulos locales: {subtitle_path}")

elif SUBTITLE_SOURCE == "youtube_auto" and USE_YOUTUBE:
    # Sistema inteligente
    manager = IntelligentSubtitleManager(YOUTUBE_URL, TARGET_LANGUAGE)
    subtitle_path = manager.get_subtitle_intelligent(
        use_whisper_translation=USE_WHISPER_TRANSLATION
    )

elif SUBTITLE_SOURCE == "youtube_manual" and USE_YOUTUBE:
    # Solo subtítulos manuales de YouTube
    print(f"📥 Buscando subtítulos manuales en YouTube ({TARGET_LANGUAGE})...")
    manager = IntelligentSubtitleManager(YOUTUBE_URL, TARGET_LANGUAGE)
    subtitle_path = manager.download_youtube_subtitle(TARGET_LANGUAGE, 'manual')
    
    if not subtitle_path:
        print("⚠️  No hay subtítulos manuales, usando sistema inteligente...")
        subtitle_path = manager.get_subtitle_intelligent(USE_WHISPER_TRANSLATION)

elif SUBTITLE_SOURCE == "whisper":
    # Solo Whisper
    print(f"🎤 Transcribiendo con Whisper ({WHISPER_MODEL})...")
    manager = IntelligentSubtitleManager(YOUTUBE_URL if USE_YOUTUBE else "", TARGET_LANGUAGE)
    
    if USE_WHISPER_TRANSLATION:
        subtitle_path = manager.transcribe_with_whisper_translation(
            video_path, TARGET_LANGUAGE, WHISPER_MODEL
        )
    else:
        subtitle_path = manager.transcribe_with_whisper(
            video_path, TARGET_LANGUAGE, WHISPER_MODEL
        )

else:
    raise ValueError(f"Configuración inválida: SUBTITLE_SOURCE={SUBTITLE_SOURCE}")

# Verificar archivos
if not os.path.exists(video_path):
    raise FileNotFoundError(f"No se encuentra el video: {video_path}")
if not os.path.exists(subtitle_path):
    raise FileNotFoundError(f"No se encuentran los subtítulos: {subtitle_path}")

print("\n" + "="*80)
print("✅ ARCHIVOS LISTOS")
print("="*80)
print(f"📹 Video: {video_path}")
print(f"📝 Subtítulos: {subtitle_path}")
print("="*80 + "\n")

# PASO 3: Procesar video con TTS
processor = VideoTTSProcessor(
    video_path=video_path,
    subtitle_path=subtitle_path,
    output_path=OUTPUT_PATH,
    tts_engine=TTS_ENGINE,
    tts_voice=EDGE_VOICE if TTS_ENGINE == "edge" else None,
    tts_lang=GTTS_LANG if TTS_ENGINE == "gtts" else TARGET_LANGUAGE
)

processor.process()

# RESULTADO
print("\n" + "="*80)
print("🎉 ¡VIDEO PROCESADO EXITOSAMENTE!")
print("="*80)
print(f"📁 Archivo: {OUTPUT_PATH}")
print(f"💾 Tamaño: {os.path.getsize(OUTPUT_PATH) / 1024 / 1024:.2f} MB")
print(f"📝 Subtítulos usados: {subtitle_path}")
print("\n📥 Para descargar: Click derecho en el archivo → Download")
print("="*80)

## 👁️ Paso 8: Previsualizar Video

In [ ]:
# Mostrar el video resultante
print("🎬 Video resultante:")
display(Video(OUTPUT_PATH, width=800))

## 📥 Paso 9: Descargar Resultado

In [ ]:
from google.colab import files

print("📥 Iniciando descarga...")
files.download(OUTPUT_PATH)

# También descargar los subtítulos si quieres revisarlos
if subtitle_path and os.path.exists(subtitle_path):
    print("📥 Descargando subtítulos usados...")
    files.download(subtitle_path)

## 🎤 EXTRA: Ver Voces Disponibles

In [ ]:
# Mostrar voces recomendadas
voices = EdgeTTSEngine.get_recommended_voices()

print("\n" + "="*80)
print("🎤 VOCES RECOMENDADAS POR IDIOMA")
print("="*80)

for lang, voice_list in voices.items():
    print(f"\n{lang.upper()}:")
    for voice_id, description in voice_list:
        print(f"  • {voice_id}")
        print(f"    {description}")

print("\n" + "="*80)

## 📋 Ejemplos de Configuración

### Ejemplo 1: YouTube con Sistema Inteligente
```python
USE_YOUTUBE = True
YOUTUBE_URL = "https://www.youtube.com/watch?v=..."
SUBTITLE_SOURCE = "youtube_auto"  # Intenta todo automáticamente
TARGET_LANGUAGE = "es"
TTS_ENGINE = "edge"
EDGE_VOICE = "es-ES-AlvaroNeural"
```

### Ejemplo 2: Video Local con Whisper
```python
USE_YOUTUBE = False
LOCAL_VIDEO_PATH = "/content/mi_video.mp4"
SUBTITLE_SOURCE = "whisper"
WHISPER_MODEL = "base"
TARGET_LANGUAGE = "es"
TTS_ENGINE = "edge"
```

### Ejemplo 3: YouTube con Traducción
```python
USE_YOUTUBE = True
YOUTUBE_URL = "https://www.youtube.com/watch?v=..."  # Video en inglés
SUBTITLE_SOURCE = "youtube_auto"  # Descargará inglés y traducirá
TARGET_LANGUAGE = "es"  # Traducir al español
TTS_ENGINE = "edge"
EDGE_VOICE = "es-MX-DaliaNeural"  # Voz mexicana
```

### Ejemplo 4: Solo Subtítulos Manuales
```python
USE_YOUTUBE = True
YOUTUBE_URL = "https://www.youtube.com/watch?v=..."
SUBTITLE_SOURCE = "youtube_manual"  # Solo subtítulos humanos
TARGET_LANGUAGE = "de"  # Alemán
TTS_ENGINE = "edge"
EDGE_VOICE = "de-DE-KatjaNeural"
```